# Superstore Sales Workflow Tutorial

This notebook walks through cleaning the Superstore dataset, designing a relational structure, creating an SQLite database, and running example SQL queries and visualizations.

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

## 1️) Load and Inspect the Dataset

In [2]:
# reading error codes + research is how you can learn if there is different encoding, as seen below
df = pd.read_csv('../data/Superstore.csv', encoding='ISO-8859-1')
df.head()
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

### Double check some datatypes, and for nulls and duplicates:

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

This data is incredibly clean - we can see there are no null values and no duplicates. However, there are some dates that could be saved in a better format.

## 2) Data Cleaning

In [6]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

In [7]:
df.dtypes

Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

In [8]:
df.Segment.unique()

array(['Consumer', 'Corporate', 'Home Office'], dtype=object)

In [9]:
df.rename(columns={"Segment":"customer_category"}, inplace=True)
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,customer_category,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2013-152156,2013-11-09,2013-11-12,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,2013-11-09,2013-11-12,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,2013-06-13,2013-06-17,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


With real data, it is very likely more cleaning and wrangling would be required. 

## 3) Build Relational Tables

In [10]:
conn = sqlite3.connect('../data/superstore.db')

customers_df = df[['Customer ID','Customer Name','customer_category','City','State','Region', 'Postal Code']].drop_duplicates().rename(columns={
    'Customer ID':'customer_id',
    'Customer Name':'customer_name',
    'City':'city',
    'State':'state',
    'Region':'region',
    'Postal Code': 'zipcode'
})

products_df = df[['Product ID','Category','Sub-Category','Product Name']].drop_duplicates().rename(columns={
    'Product ID':'product_id',
    'Category':'category',
    'Sub-Category':'subcategory',
    'Product Name':'product_name'
})

orders_df = df[['Order ID','Order Date','Ship Date','Ship Mode','Customer ID']].drop_duplicates().rename(columns={
    'Order ID':'order_id',
    'Order Date':'order_date',
    'Ship Date':'ship_date',
    'Ship Mode':'ship_mode',
    'Customer ID':'customer_id'
})

order_details_df = df[['Order ID','Product ID','Sales','Quantity','Discount','Profit']].rename(columns={
    'Order ID':'order_id',
    'Product ID':'product_id'
})


customers_df.to_sql('customers', conn, index=False, if_exists='replace')
products_df.to_sql('products', conn, index=False, if_exists='replace')
orders_df.to_sql('orders', conn, index=False, if_exists='replace')
order_details_df.to_sql('order_details', conn, index=False, if_exists='replace')
conn.commit()

In [11]:
for name, df in [
    ('customers', customers_df),
    ("products", products_df),
    ("orders", orders_df),
    ("order_details", order_details_df)
]:
    df.to_csv(f'../data/{name}.csv')

## 4) Make Some Queries!

### Which customer segments bring in the most revenue and profit?

In [12]:
query1 = """ SELECT 
    c.customer_category,
    ROUND(SUM(od.sales), 2) AS total_sales,
    ROUND(SUM(od.profit), 2) AS total_profit
FROM order_details od
JOIN orders o ON od.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_category
ORDER BY total_sales DESC;
"""
result1 = pd.read_sql(query1, conn)
result1

,customer_category,total_sales,total_profit
0,Consumer,8380282.43,973085.86
1,Corporate,5044899.83,660287.69
2,Home Office,2964459.36,434344.01


### What are the top 10 customers by total sales?

In [14]:
query2 = """ 

SELECT 
    c.customer_name as customer, 
    ROUND(SUM(od.sales), 2) as total_sales
FROM customers c
JOIN orders o 
    ON c.customer_id = o.customer_id
JOIN order_details od
    ON o.order_id = od.order_id
GROUP BY customer
ORDER BY total_sales desc
LIMIT 10

"""
result2 = pd.read_sql(query2, conn)
result2

,customer,total_sales
0,Ken Lonsdale,155927.52
1,Sanjit Engle,134303.82
2,Clay Ludtke,130566.55
3,Adrian Barton,130262.14
4,Sanjit Chand,127281.01
5,Sean Miller,125215.25
6,Edward Hooks,123730.56
7,Greg Tran,118201.20
8,Seth Vernon,114709.50
9,John Lee,107799.15


### Which product categories and subcategories are the most profitable overall?

In [27]:
query3 = """ 

WITH subcategory_profits AS (
    SELECT 
        p.category,
        p.subcategory,
        ROUND(SUM(od.profit), 2) AS total_profit
    FROM products p 
    JOIN order_details od 
        ON p.product_id = od.product_id
    GROUP BY p.category, p.subcategory
),
category_totals AS (
    SELECT 
        category,
        ROUND(SUM(total_profit), 2) AS category_total_profit
    FROM subcategory_profits
    GROUP BY category
)
SELECT 
    s.category,
    s.subcategory,
    s.total_profit,
    c.category_total_profit
FROM subcategory_profits s
JOIN category_totals c 
    ON s.category = c.category
ORDER BY s.total_profit DESC;

"""
result3 = pd.read_sql(query3, conn)
result3

,category,subcategory,total_profit,category_total_profit
0,Technology,Copiers,55617.82,153415.70
1,Technology,Accessories,48359.05,153415.70
2,Technology,Phones,46936.19,153415.70
3,Office Supplies,Paper,36994.53,126113.33
4,Office Supplies,Binders,30373.20,126113.33
5,Furniture,Chairs,26707.65,20098.89
6,Office Supplies,Storage,21408.70,126113.33
7,Office Supplies,Appliances,18514.49,126113.33
8,Furniture,Furnishings,14569.59,20098.89
9,Office Supplies,Envelopes,6964.18,126113.33


### What is the average discount given per category?

In [29]:
query4 = """ 

SELECT p.category as category, round(avg(od.discount),2) as avg_discount
FROM products p
JOIN order_details od 
    ON p.product_id = od.product_id
GROUP BY category
ORDER BY avg_discount desc

 """
result4 = pd.read_sql(query4, conn)
result4

,category,avg_discount
0,Furniture,0.17
1,Office Supplies,0.16
2,Technology,0.13


### Which states or regions contribute the most to total sales?

In [35]:
query5 = """

WITH state_sales AS (
    SELECT 
        c.state,
        ROUND(SUM(od.sales), 2) AS total_sales
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_details od ON o.order_id = od.order_id
    GROUP BY c.state
)
SELECT 
    state,
    total_sales,
    ROUND(100.0 * total_sales / SUM(total_sales) OVER (), 2) AS pct_total_sales
FROM state_sales
ORDER BY total_sales DESC;


"""
result5 = pd.read_sql(query5, conn)
result5

,state,total_sales,pct_total_sales
0,California,3314837.69,20.23
1,New York,1859365.37,11.34
2,Texas,1547925.76,9.44
3,Pennsylvania,886105.68,5.41
4,Washington,862298.69,5.26
5,Illinois,843352.76,5.15
6,Ohio,707747.69,4.32
7,Florida,635637.78,3.88
8,North Carolina,437938.43,2.67
9,Michigan,432428.02,2.64


### How many orders were shipped late (where Ship Date > Order Date + 3 days)?

In [38]:
query6 = """

WITH order_stats AS (
    SELECT 
        COUNT(*) AS total_orders,
        COUNT(CASE WHEN ship_date > DATE(order_date, '+3 days') THEN 1 END) AS late_orders
    FROM orders
)
SELECT 
    late_orders,
    total_orders,
    ROUND(100.0 * late_orders / total_orders, 2) AS pct_late
FROM order_stats;

"""
result6 = pd.read_sql(query6, conn)
result6

,late_orders,total_orders,pct_late
0,3902,5009,77.9


### Which month or year had the highest total sales?

In [44]:
query7 = """

SELECT 
    STRFTIME('%Y', o.order_date) AS year,
    ROUND(SUM(od.sales), 2) AS total_sales
FROM orders o
JOIN order_details od
    ON o.order_id = od.order_id
GROUP BY year
ORDER BY total_sales DESC


"""
result7 = pd.read_sql(query7, conn)
result7

,year,total_sales
0,2014,733947.02
1,2013,608473.83
2,2011,484247.50
3,2012,470532.51


### What are the most frequently ordered products?

In [46]:
query8 = """

SELECT 
    p.product_name as product_name,
    COUNT(od.product_id) as number_orders
FROM products p
JOIN order_details od 
    ON p.product_id = od.product_id 
GROUP BY product_name
ORDER BY number_orders desc

"""
result8 = pd.read_sql(query8, conn)
result8

,product_name,number_orders
0,Staples,227
1,Avery Non-Stick Binders,20
2,Xerox 1908,19
3,Xerox 1881,19
4,Logitech P710e Mobile Speakerphone,18
...,...,...
1836,Avery 484,1
1837,Avaya IP Phone 1140E VoIP phone,1
1838,Acco Glide Clips,1
1839,AT&T EL51110 DECT,1


### Which customers placed more than 5 orders total?

In [49]:
query9 = """

WITH customer_order_counts as (
    SELECT 
        c.customer_name as customer_name,
        COUNT(*) as total_orders
    FROM customers c
    JOIN orders o 
        ON c.customer_id = o.customer_id
    GROUP BY customer_name 
    )
SELECT 
    customer_name, 
    total_orders
FROM customer_order_counts
WHERE total_orders > 5
ORDER BY total_orders DESC

"""
result9 = pd.read_sql(query9, conn)
result9

,customer_name,total_orders
0,Emily Phan,289
1,Erin Ashbrook,169
2,Joel Eaton,169
3,Sally Hughsby,169
4,Zuschuss Carroll,169
...,...,...
742,Shirley Schmidt,9
743,Susan Gilcrest,9
744,Tim Taslimi,9
745,Tony Molinari,9


### What’s the profit margin (profit ÷ sales) by category or segment?

In [54]:
query10 = """
WITH category_data as (
    SELECT 
        p.category as category,
        SUM(od.profit) as total_profit,
        SUM(od.sales) as total_sales
    FROM products p
    JOIN order_details od 
        ON p.product_id = od.product_id
    GROUP BY category
    )
SELECT 
    category, 
    SUM(total_profit / total_sales) as profit_margin
FROM category_data
GROUP BY category
ORDER BY profit_margin DESC;

"""
result10 = pd.read_sql(query10, conn)
result10

,category,profit_margin
0,Technology,0.171676
1,Office Supplies,0.171176
2,Furniture,0.026298


### 5) Visualize it!

From your ten results, select at least 4 to utilize in a meaningful visualization. 

In [ ]:
# example using result1
plt.figure(figsize=(10,6))
colors = ["#a7adaa", "#a7adaa", "#f74141" ]
sns.barplot(result1, x='customer_category', y='total_sales', palette=colors, hue='customer_category', legend=False)
plt.title('Small Offices, Smaller Sales: The Home Office Gap',fontsize=18)
plt.xlabel(None)
plt.ylabel('Total Sales ($USD)')

#for those wondering how to get rid of scientific notation! Set style to plain:
plt.ticklabel_format(style='plain', axis='y')

#mticker is a great library for helping tick mark formatting, such as adding commas and dollar signs in this case
ax = plt.gca()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

#using the seaborn library, despining is way easier!
sns.despine()
plt.show()


In [ ]:
# visualization one code here

In [ ]:
# visualization two code here

In [ ]:
# visualization three code here

In [ ]:
# visualization four code here